# Idealized Digital Transmission System (4-PAM)

*Python/Colab conversion of the MATLAB script `idsys.m`*
(the capstone example from **Software Receiver Design**, Johnson, Sethares & Klein).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oalnaseri/CommSystems_Course/blob/main/04_idsys.ipynb)

> **How to run:** Click the **Open in Colab** badge above, then run every cell top-to-bottom with `Shift + Enter`, or use **Runtime -> Run all**. No login needed — just click and run.

**What this does:** it sends a *text message* through a complete communication system and recovers it:

```
text -> 4-PAM symbols -> pulse shaping -> modulate -> [channel]
     -> demodulate -> low-pass filter -> matched filter -> downsample
     -> decide symbols -> text back
```

Each stage is plotted, and at the end we measure the **symbol error rate** and print the **reconstructed message**.

No installation needed — `numpy`, `scipy`, `matplotlib` and `plotly` are pre-installed in Colab.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import remez, lfilter          # remez = MATLAB firpm; lfilter = MATLAB filter
from scipy.signal.windows import hamming          # MATLAB hamming(M)

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plotly'])
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 4)

## 2. Textbook helper functions (re-implemented in Python)

`idsys.m` relies on several helpers from the *Software Receiver Design* toolbox that aren't part of
standard Python. We re-create them here:

| MATLAB helper | Purpose | Python here |
|---|---|---|
| `letters2pam` | text → 4-level symbols (±1, ±3) | `letters2pam` |
| `pam2letters` | 4-level symbols → text | `pam2letters` |
| `quantalph`   | quantize to nearest alphabet symbol | `quantalph` |
| `pow`         | signal power = mean of squares | `pow_` |
| `plotspec`    | waveform + magnitude spectrum | `plotspec` |

Each ASCII character (0–255) is written as **four base-4 digits**, and each digit `d ∈ {0,1,2,3}`
maps to a **4-PAM symbol** `2d − 3 ∈ {−3, −1, 1, 3}`.

In [ ]:
def letters2pam(s):
    """Encode an ASCII string as a 4-PAM sequence in {-3,-1,1,3} (4 symbols/char)."""
    x = []
    for ch in s:
        v = ord(ch)
        digits = [(v // 64) % 4, (v // 16) % 4, (v // 4) % 4, v % 4]  # base-4, MSB first
        x += [2 * d - 3 for d in digits]
    return np.array(x, dtype=float)


def pam2letters(seq):
    """Decode a 4-PAM sequence back into an ASCII string (inverse of letters2pam)."""
    seq = np.asarray(seq)
    out = []
    n = (len(seq) // 4) * 4                      # only decode complete 4-symbol groups
    for k in range(0, n, 4):
        d = [int(round((s + 3) / 2)) for s in seq[k:k+4]]  # symbol -> digit 0..3
        d = [min(max(v, 0), 3) for v in d]                 # clamp to valid range
        out.append(chr((d[0]*64 + d[1]*16 + d[2]*4 + d[3]) % 256))
    return ''.join(out)


def quantalph(x, alphabet):
    """Quantize each value of x to the nearest element of alphabet."""
    x = np.asarray(x).reshape(-1, 1)
    a = np.asarray(alphabet).reshape(1, -1)
    idx = np.argmin(np.abs(x - a), axis=1)
    return np.asarray(alphabet).ravel()[idx]


def pow_(x):
    """Signal power: mean of squares (MATLAB pow.m)."""
    x = np.asarray(x)
    return np.sum(x**2) / len(x)


def plotspec(x, Ts, flim=None, title=''):
    """Waveform + magnitude spectrum (Python port of plotspec.m), with optional zoom."""
    x = np.asarray(x); N = len(x)
    t = Ts * np.arange(1, N + 1)
    ssf = np.arange(-N/2, N/2) / (Ts * N)
    fxs = np.fft.fftshift(np.fft.fft(x))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6))
    ax1.plot(t, x); ax1.set_xlabel('seconds'); ax1.set_ylabel('amplitude')
    ax1.set_title(title or 'Waveform'); ax1.grid(True)
    ax2.plot(ssf, np.abs(fxs)); ax2.set_xlabel('frequency (Hz)')
    ax2.set_ylabel('magnitude'); ax2.set_title('Magnitude spectrum'); ax2.grid(True)
    if flim is not None:
        lo, hi = (-flim, flim) if np.isscalar(flim) else (flim[0], flim[1])
        ax2.set_xlim(lo, hi)
    fig.tight_layout(); plt.show()

Quick self-check that encoding then decoding returns the original text exactly:

In [ ]:
_test = '01234 I wish I were an Oscar Meyer wiener 56789'
assert pam2letters(letters2pam(_test)) == _test
print('letters2pam / pam2letters round-trip: OK')

## 3. Transmitter

The text becomes 4-PAM symbols, which are **upsampled** (spread out by 100×) and **pulse-shaped** with a Hamming blip to form the baseband waveform `x`.

In [ ]:
# encode text as a T-spaced 4-PAM sequence
s = '01234 I wish I were an Oscar Meyer wiener 56789'
m = letters2pam(s)
N = len(m)                          # number of symbols

M = 100                             # oversampling factor (samples per symbol)
mup = np.zeros(N * M)               # upsampled T/M-spaced sequence
mup[::M] = m                        # place one symbol impulse every M samples

p = hamming(M)                      # Hamming 'blip' pulse of width M
x = lfilter(p, 1, mup)             # convolve pulse shape with data

print(f'message: {N} symbols  ->  waveform x: {len(x)} samples')

**Baseband waveform and its spectrum** :

In [ ]:
plotspec(x, 1/M, title='Baseband pulse-shaped signal x')

### Modulate onto the carrier

In [ ]:
t = np.arange(1, len(x) + 1) / M    # T/M-spaced time vector
fc = 20                             # carrier frequency
c = np.cos(2*np.pi*fc*t)            # carrier
r = c * x                           # transmitted (modulated) signal

plotspec(r, 1/M, title='Transmitted signal r = c * x')

## 4. Receiver

We mix with a synchronized carrier, then low-pass filter to bring the message back to baseband. (As before, MATLAB's `firpm` → SciPy's `remez` with `fs=2` so the Nyquist-normalised band edges match.)

In [ ]:
c2 = np.cos(2*np.pi*fc*t)           # synchronized cosine for demodulation
x2 = r * c2                         # demodulated received signal

fl = 50                             # LPF order
fbe = [0, 0.1, 0.2, 1]              # band edges (normalised to Nyquist = 1)
damps = [1, 0]                      # pass-band gain 1, stop-band gain 0
b = remez(fl + 1, fbe, damps, fs=2) # LPF impulse response (firpm equivalent)

x3 = 2 * lfilter(b, 1, x2)          # low-pass filter and scale by 2

plotspec(x3, 1/M, title='Recovered baseband signal x3')

### Matched filter + downsample to symbol rate

The **matched filter** correlates the incoming waveform with the known pulse shape to maximize the
signal-to-noise ratio, then we **downsample** — picking one sample per symbol at the right timing
offset (which accounts for the combined filter delays).

In [ ]:
# matched filter: time-reversed pulse, normalised so the peak correlation is 1
matched = np.flip(p) / (pow_(p) * M)
y = lfilter(matched, 1, x3)

# downsample: MATLAB y(0.5*fl+M : M : N*M) is 1-indexed; shift by -1 for Python
start = int(0.5 * fl + M)                       # first symbol-sample (1-indexed)
idx = np.arange(start, N * M + 1, M) - 1        # convert to 0-indexed
z = y[idx]                                      # soft decisions (one per symbol)

print(f'downsampled to {len(z)} soft decisions')

**Soft decisions** — each dot is one received symbol *before* the decision. Clean clusters near −3, −1, +1, +3 mean the system is working well.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.arange(1, len(z) + 1), z, '.')
for lvl in [-3, -1, 1, 3]:
    ax.axhline(lvl, color='r', ls='--', lw=0.7, alpha=0.6)
ax.set_xlabel('symbol index'); ax.set_ylabel('soft decision value')
ax.set_title('Soft decisions (dots should cluster on ±1, ±3)'); ax.grid(True)
fig.tight_layout(); plt.show()

## 5. Decision device & performance assessment

* **Cluster variance** — how tightly the soft decisions hug the ideal levels (smaller = better).
* **Percent symbol error** — fraction of symbols decided incorrectly.
* **Reconstructed message** — the decoded text.

In [ ]:
mprime = quantalph(z, [-3, -1, 1, 3])       # hard decisions (nearest symbol)

cvar = np.sum((mprime - z)**2) / len(mprime) # cluster variance
lmp = len(mprime)
pererr = 100 * np.sum(np.abs(np.sign(mprime - m[:lmp]))) / lmp  # % symbol error

print(f'cluster variance   : {cvar:.6f}')
print(f'percent symbol error: {pererr:.2f} %')

reconstructed_message = pam2letters(mprime)
print()
print('original     :', repr(s))
print('reconstructed:', repr(reconstructed_message))

> The reconstructed text should match the original. The final character may be dropped because the
> last symbol is lost to the filter delay (we recover `N-1` of `N` symbols) — this is expected 

## 6. Notes & tips

- **The whole point:** this shows every block of a real digital link — source coding, pulse shaping, modulation, demodulation, filtering, matched filtering, timing/downsampling, and symbol decisions — and proves it by recovering readable text with **0% error** in the ideal (noiseless) case.
- **`letters2pam` / `pam2letters`:** each character = 4 base-4 digits, each digit → one of {−3,−1,1,3}.
- **Matched filter normalisation:** `fliplr(p)/(pow(p)*M)` equals `flip(p)/sum(p**2)`, which sets the peak of the pulse autocorrelation to 1.
- **Timing offset:** `z = y[0.5*fl + M ... ]` skips the combined delay of the LPF and pulse/matched filters so we sample each symbol at its peak. Remember MATLAB is 1-indexed, so we subtract 1 in Python.
- **Try it yourself:** change the message string, the carrier `fc`, the oversampling `M`, or the filter order `fl`, and watch the soft-decision clusters and error rate respond. Add noise to `r` (e.g. `r += 0.2*np.random.randn(len(r))`) to see a realistic channel!
